# Runner (Colab / Kaggle)

Clones the repo, installs LeRobot + LIBERO, and runs one experiment config. Results are appended to a CSV on Drive (Colab) or in `/kaggle/working` (Kaggle); the script skips episodes already in the CSV, so re-running this notebook after a disconnect resumes the run.

All numbers in the report were produced on an Apple M1 laptop; this notebook is here to scale the same configs up on a T4.

In [ ]:
CONFIG = "configs/baseline_spatial.yaml"   # experiment to run
SHARD = "0/1"                               # e.g. "0/2" on Colab and "1/2" on Kaggle
REPO = "https://github.com/AnaisAzouaoui/vla-stress-test.git"

import os, subprocess
ON_KAGGLE = os.path.exists("/kaggle")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/vla-stress-results"
else:
    OUT_DIR = "/kaggle/working/results"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
!git clone -q $REPO repo || (cd repo && git pull -q)
%cd repo
!pip install -q -e ".[libero,dev]"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

Sanity check with the official evaluation script (2 episodes). The rename map is needed because the checkpoint names its cameras `camera1/camera2`.

In [ ]:
!lerobot-eval --policy.path=lerobot/smolvla_libero --env.type=libero --env.task=libero_goal --env.task_ids="[0]" \
  --eval.batch_size=1 --eval.n_episodes=2 --policy.device=cuda --output_dir=outputs/sanity \
  --rename_map='{"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}'

In [ ]:
import yaml, pathlib
name = yaml.safe_load(open(CONFIG))["name"]
out = f"{OUT_DIR}/{name}.shard{SHARD.replace('/', 'of')}.csv"
!python -m vla_stress.evaluate --config $CONFIG --shard $SHARD --out $out --device cuda

In [ ]:
import pandas as pd
df = pd.read_csv(out)
df.groupby(["condition", "intensity"])["success"].agg(["mean", "count"])